# EDA: conocer los datos de mesociclones en Chile

Análisis exploratorio de las tres fuentes del proyecto: el **catálogo de tornados**, el reanálisis **ERA5** (superficie, niveles de presión e índices de tiempo severo) y las imágenes del satélite **GOES**.

El cuaderno lee los recortes que dejan los scripts de `scripts/pipeline/` en `data/crops/<región>/`, siguiendo las mismas convenciones que el visor (`scripts/viewer/viewer_app/data.py`). Si todavía no hay datos descargados, cada sección lo avisa y explica cómo generarlos; el catálogo (sección 1) funciona en cuanto exista `data/tornados_chile.xlsx`.

> Las horas están en **UTC**. En otoño, la hora de Chile continental es UTC − 4.

**Trampas del dominio** (ver `docs/mesociclones_chile.md` y `AGENTS.md`): estamos en el **hemisferio sur**, así que la helicidad tornádica (SRH) es **negativa** y la supercélula favorecida es la *left-mover*; el STP se calcula en su rama *left-mover* (umbral chileno `≤ −0,3`). El régimen es **HSLC** (baja CAPE, alta cizalladura), por lo que los umbrales norteamericanos no sirven. El catálogo solo tiene **positivos**: que un día no tenga registro no prueba que no hubo tornado.

## 0. Preparación

Añadimos `scripts/` al `sys.path` para reutilizar `common` y `store` del proyecto (igual que hacen los scripts del pipeline), y fijamos la raíz de datos. Para apuntar a otra raíz —por ejemplo el bucket del equipo o una carpeta montada desde la EC2— exporta `MESO_DATA_ROOT` antes de abrir Jupyter.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# scripts/ no es un paquete instalable: se agrega a sys.path para importar common y store,
# igual que hacen los scripts del pipeline. Buscamos la raíz del repo subiendo carpetas hasta
# encontrar scripts/common.py, para que el cuaderno funcione desde notebooks/ o desde la raíz.
def _find_repo() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "common.py").exists():
            return base
    raise FileNotFoundError("No encuentro scripts/common.py: abre el cuaderno dentro del repositorio.")

REPO = _find_repo()
SCRIPTS = str(REPO / "scripts")
if SCRIPTS in sys.path:
    sys.path.remove(SCRIPTS)
sys.path.insert(0, SCRIPTS)  # al frentUnae, para no chocar con paquetes de PyPI del mismo nombre

import common  # noqa: E402
from store import Store  # noqa: E402

assert Path(common.__file__).parent == REPO / "scripts", f"common equivocado: {common.__file__}"

REGION = "south-central-austral"  # la región que descarga el pipeline por defecto
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

print("repositorio :", REPO)
print("raíz de datos:", common.DATA.root)
print("recortes en  :", common.CROPS.root)
print("existe data/ :", common.DATA.exists())

In [ ]:
# Qué regiones y días hay descargados. Las rutas siguen el patrón de data.py del visor.
regions = sorted(p.name for p in common.CROPS.glob("*") if p.is_dir()) if common.CROPS.exists() else []
print("regiones con recortes:", regions or "(ninguna todavía)")

if REGION not in regions and regions:
    REGION = regions[0]
    print(f"-> usando la región disponible: {REGION}")

import re

def dates_in(source: str) -> list[str]:
    """Fechas AAAAMMDD presentes para un tipo de dato de la región elegida."""
    base = common.CROPS.child(REGION, source)
    if not base.exists():
        return []
    out = []
    for p in base.glob("*"):
        m = re.findall(r"\d{8}", p.name)
        if m:
            out.append(m[-1])
    return sorted(set(out))

inventory = {src: dates_in(src) for src in ["era5_sfc", "era5_pl", "era5_indices", "goes_latlon"]}
for src, ds in inventory.items():
    print(f"{src:14s}: {len(ds)} día(s) {ds}")

# Día de trabajo por defecto: el primero con índices ERA5, o 20190530 (el caso del brote).
available = sorted(set().union(*inventory.values())) if any(inventory.values()) else []
DAY = (inventory["era5_indices"] or available or ["20190530"])[0]
print("\ndía de trabajo:", DAY)

## 1. El catálogo de tornados y trombas

`data/tornados_chile.xlsx`, hoja `Final_Table`: 102 eventos catalogados entre 1554 y 2025 (Bastías-Curivil et al., figshare, CC BY 4.0). Es la **etiqueta** del problema. Columnas clave:

- `Date (Gregorian Calendar)`, `Latitude`, `Longitude`, `Hour (UTC)` — cuándo y dónde (la hora cruza con GOES).
- `Tornado`, `Waterspout` — las dos vías físicas (una fila puede ser ambas).
- `Register` — confianza de la etiqueta (1 = registro confirmado).
- `Max. EF Rating` — intensidad.

**Solo contiene positivos**: no hay días de tormenta sin tornado, así que cualquier modelo necesita construir sus propios negativos.

In [ ]:
CATALOG = common.DATA.child("tornados_chile.xlsx")

def coordinate(v) -> float:
    """Coordenada del catálogo a número (algunas filas vienen como texto con comas)."""
    s = str(v).strip().rstrip(",").strip()
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    m = re.match(r"-?\d+(\.\d+)?", s)
    return float(m.group()) if m else np.nan

if not CATALOG.exists():
    raise FileNotFoundError(
        f"Falta {CATALOG.root}. Descárgalo con:\n"
        "  curl -L -o data/tornados_chile.xlsx https://ndownloader.figshare.com/files/56272544"
    )

with CATALOG.fs.open(CATALOG.root, "rb") as f:
    cat = pd.read_excel(f, sheet_name="Final_Table")

cat["date"] = pd.to_datetime(cat["Date (Gregorian Calendar)"], errors="coerce")
cat["lat"] = cat["Latitude"].map(coordinate)
cat["lon"] = cat["Longitude"].map(coordinate)
cat["year"] = cat["date"].dt.year
cat["month"] = cat["date"].dt.month
print(f"{len(cat)} eventos; columnas: {list(cat.columns)[:12]} ...")
cat[["date", "Location", "lat", "lon", "Tornado", "Waterspout", "Register", "Hour (UTC)", "Max. EF Rating"]].head(10)

In [ ]:
# Tipo de evento, confianza e intensidad.
tornado = cat["Tornado"].fillna(0).astype(int) == 1
waterspout = cat["Waterspout"].fillna(0).astype(int) == 1
kind = np.select([tornado & waterspout, tornado, waterspout],
                 ["tornado y tromba", "tornado", "tromba"], "sin clasificación")
print("Por tipo:")
print(pd.Series(kind).value_counts().to_string())
print("\nConfianza (Register):")
print(cat["Register"].fillna(0).astype(int).map({1: "confirmado", 0: "débil"}).value_counts().to_string())
print("\nIntensidad (Max. EF Rating):")
print(cat["Max. EF Rating"].fillna("?").astype(str).str.replace(r"\.0$", "", regex=True).value_counts().to_string())

In [ ]:
# Distribución temporal: por año (sesgo de reporte) y por mes (estacionalidad).
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
yrs = cat["year"].dropna().astype(int)
axes[0].hist(yrs[yrs >= 1950], bins=range(1950, 2027, 5), color="#b2182b", edgecolor="white")
axes[0].set(title="Eventos por año (desde 1950)", xlabel="año", ylabel="nº eventos")

months = cat["month"].dropna().astype(int)
order = [1,2,3,4,5,6,7,8,9,10,11,12]
counts = months.value_counts().reindex(order, fill_value=0)
axes[1].bar([f"{m:02d}" for m in order], counts.values, color="#2166ac")
axes[1].set(title="Estacionalidad (mes)", xlabel="mes", ylabel="nº eventos")
plt.tight_layout(); plt.show()
print("Dos tercios de los eventos son de los últimos ~11 años: refleja mejor reporte, no más tornados.")

In [ ]:
# Dónde ocurren: mapa simple lat/lon (triángulos = tornados, puntos = trombas).
fig, ax = plt.subplots(figsize=(5.5, 8))
m = cat.dropna(subset=["lat", "lon"])
ax.scatter(m.loc[waterspout.reindex(m.index, fill_value=False), "lon"],
           m.loc[waterspout.reindex(m.index, fill_value=False), "lat"],
           s=25, c="#2166ac", marker="o", label="tromba", alpha=0.7)
ax.scatter(m.loc[tornado.reindex(m.index, fill_value=False), "lon"],
           m.loc[tornado.reindex(m.index, fill_value=False), "lat"],
           s=35, c="#b2182b", marker="^", label="tornado", alpha=0.8)
ax.set(title="Eventos catalogados", xlabel="longitud", ylabel="latitud",
       xlim=(-76, -66), ylim=(-45, -18))
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("La mayoría se concentra en el centro-sur, entre ~ -34° y -40° de latitud.")

In [ ]:
# El caso de referencia: el brote del 30-31 de mayo de 2019 (cinco tornados el día 30).
brote = cat[(cat["date"] >= "2019-05-30") & (cat["date"] <= "2019-05-31")]
cols = ["date", "Location", "lat", "lon", "Hour (UTC)", "Tornado", "Waterspout", "Max. EF Rating", "Register"]
brote[[c for c in cols if c in brote.columns]]

## 2. Inventario de los datos descargados

Antes de abrir ERA5 y GOES, resumimos qué hay para el día de trabajo. Si el inventario está vacío, genera un día con:

```bash
uv run python scripts/pipeline/download_era5.py --start 2019-05-30
uv run python scripts/pipeline/download_goes.py  --start 2019-05-30
uv run python scripts/pipeline/process_goes.py   --start 2019-05-30
uv run python scripts/pipeline/process_era5.py   --start 2019-05-30
```

In [ ]:
def nc(source: str, prefix: str, day: str) -> Store | None:
    """Ruta del NetCDF <source>/<prefix>_<day>.nc si existe."""
    p = common.CROPS.child(REGION, source, f"{prefix}_{day}.nc")
    return p if p.exists() else None

sfc_p = nc("era5_sfc", "era5_sfc", DAY)
pl_p = nc("era5_pl", "era5_pl", DAY)
idx_p = nc("era5_indices", "era5_idx", DAY)
goes_dir = common.CROPS.child(REGION, "goes_latlon", DAY)
goes_files = goes_dir.glob("goes_*.nc") if goes_dir.exists() else []

print(f"Día {DAY}, región {REGION}:")
print("  ERA5 superficie :", "sí" if sfc_p else "no")
print("  ERA5 presión    :", "sí" if pl_p else "no")
print("  ERA5 índices    :", "sí" if idx_p else "no")
print(f"  GOES (lat/lon)  : {len(goes_files)} archivo(s)")
HAS_ERA5 = sfc_p is not None
HAS_IDX = idx_p is not None
HAS_PL = pl_p is not None
HAS_GOES = len(goes_files) > 0

## 3. ERA5 en superficie

`era5_sfc_<día>.nc` — una grilla horaria (0,25°) con viento a 10 y 100 m (`10u/10v`, `100u/100v`), temperatura y punto de rocío a 2 m (`2t`, `2d`), presiones (`sp` superficie, `msl` nivel del mar), **CAPE**, agua precipitable (`tcwv`), altura de la capa límite (`blh`) y nubosidad total (`tcc`).

In [ ]:
if not HAS_ERA5:
    print("No hay ERA5 superficie para este día. Corre download_era5.py / process_era5.py (ver sección 2).")
else:
    sfc = sfc_p.open_dataset()
    print(sfc)
    # Resumen estadístico de las variables de superficie en todo el día y la grilla.
    rows = []
    for v in sfc.data_vars:
        a = sfc[v].values.astype("float64")
        rows.append({"variable": v, "units": sfc[v].attrs.get("units", ""),
                     "min": np.nanmin(a), "mean": np.nanmean(a), "max": np.nanmax(a)})
    display(pd.DataFrame(rows).set_index("variable").round(2))

In [ ]:
if HAS_ERA5:
    # Mapas al mediodía UTC: CAPE y velocidad del viento a 10 m, con los eventos del día encima.
    t = sfc["time"].values[min(12, sfc.sizes["time"] - 1)]
    s = sfc.sel(time=t)
    spd10 = np.hypot(s["10u"], s["10v"])
    day_dt = pd.to_datetime(str(t)).date()
    ev = cat[cat["date"].dt.date == day_dt].dropna(subset=["lat", "lon"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, field, title, cmap in [(axes[0], s["cape"], "CAPE (J/kg)", "magma"),
                                    (axes[1], spd10, "Viento 10 m (m/s)", "viridis")]:
        im = field.plot(ax=ax, cmap=cmap, add_colorbar=True, cbar_kwargs={"shrink": 0.7})
        if not ev.empty:
            ax.scatter(ev["lon"], ev["lat"], c="cyan", marker="^", s=60, edgecolor="black", zorder=5)
        ax.set(title=f"{title}\n{pd.to_datetime(str(t)):%Y-%m-%d %H:%M} UTC")
    plt.tight_layout(); plt.show()
    print("Triángulos cian = tornados/trombas de ese día. Nota la CAPE modesta: régimen HSLC.")

## 4. Índices de tiempo severo (la clave del problema)

`era5_idx_<día>.nc` — calculados por `process_era5.py` con los **signos del hemisferio sur**. Los que el documento da como calibrados para Chile:

| Índice | Variable | Umbral chileno |
|---|---|---|
| Cizalladura 0–1 km | `shear_01` | `> 12 m/s` |
| VGP 0–1 km | `vgp_01` | `> 0,1 m/s²` |
| STP (rama *left-mover*) | `stp_lm` | `≤ −0,3` |
| SHERBS3 | `sherbs3` | (orientativo `≥ 1`) |

La **SRH** (`srh_01`, `srh_03`) es **negativa** en los casos tornádicos chilenos: ojo con el signo.

In [ ]:
if not HAS_IDX:
    print("No hay índices ERA5 para este día. Corre process_era5.py (ver sección 2).")
else:
    idx = idx_p.open_dataset()
    rows = []
    for v in idx.data_vars:
        a = idx[v].values.astype("float64")
        rows.append({"índice": v, "units": idx[v].attrs.get("units", ""),
                     "min": np.nanmin(a), "mediana": np.nanmedian(a), "max": np.nanmax(a)})
    display(pd.DataFrame(rows).set_index("índice").round(3))

In [ ]:
if HAS_IDX:
    # Mapas de los índices decisivos cerca de la hora de los eventos, con sus umbrales chilenos.
    ev = cat[cat["date"].dt.date == pd.to_datetime(DAY).date()].dropna(subset=["lat", "lon"])
    t = idx["time"].values[min(21, idx.sizes["time"] - 1)]  # ~tarde UTC, cuando ocurrió el brote
    g = idx.sel(time=t)
    panels = [("shear_01", "Cizalladura 0-1 km (m/s)", "viridis"),
              ("srh_03", "SRH 0-3 km (m²/s²) — negativa", "RdBu"),
              ("vgp_01", "VGP 0-1 km (m/s²)", "magma"),
              ("stp_lm", "STP left-mover — negativo", "RdBu_r")]
    fig, axes = plt.subplots(2, 2, figsize=(12, 13))
    for ax, (v, title, cmap) in zip(axes.ravel(), panels, strict=True):
        g[v].plot(ax=ax, cmap=cmap, add_colorbar=True, cbar_kwargs={"shrink": 0.7})
        if not ev.empty:
            ax.scatter(ev["lon"], ev["lat"], c="lime", marker="^", s=70, edgecolor="black", zorder=5)
        ax.set(title=title)
    fig.suptitle(f"Índices · {pd.to_datetime(str(t)):%Y-%m-%d %H:%M} UTC", y=1.0)
    plt.tight_layout(); plt.show()

In [ ]:
if HAS_IDX and not ev.empty:
    # Serie de tiempo de los índices en el punto del primer evento del día: cómo evolucionan.
    e0 = ev.iloc[0]
    col = idx.sel(latitude=e0["lat"], longitude=e0["lon"], method="nearest")
    hrs = pd.to_datetime(col["time"].values)
    fig, ax1 = plt.subplots(figsize=(11, 4))
    ax1.plot(hrs, col["shear_01"], color="#1b7837", label="shear_01 (m/s)")
    ax1.axhline(12, color="#1b7837", ls=":", alpha=0.6)
    ax1.set_ylabel("cizalladura 0-1 km (m/s)", color="#1b7837"); ax1.set_xlabel("hora UTC")
    ax2 = ax1.twinx()
    ax2.plot(hrs, col["srh_03"], color="#b2182b", label="srh_03 (m²/s²)")
    ax2.set_ylabel("SRH 0-3 km (m²/s²)", color="#b2182b")
    plt.title(f"Evolución horaria en {e0['Location']} ({e0['lat']:.2f}, {e0['lon']:.2f})")
    plt.tight_layout(); plt.show()
    print("Recuerda: SRH negativa = favorable en el hemisferio sur.")

## 5. ERA5 en niveles de presión (perfil vertical)

`era5_pl_<día>.nc` — las mismas variables en los **37 niveles** de presión: viento (`u`, `v`), velocidad vertical (`w`), temperatura (`t`), humedad específica (`q`), geopotencial (`z`), humedad relativa (`r`) y vorticidad (`vo`).

> Sobre los Andes, los niveles bajos (p. ej. 1000 hPa) quedan **bajo tierra**: aparecen como `NaN` o valores no físicos. Hay que enmascarar por presión de superficie.

In [ ]:
if not HAS_PL:
    print("No hay ERA5 niveles de presión para este día. Corre download_era5.py (ver sección 2).")
else:
    pl = pl_p.open_dataset()
    print("niveles (hPa):", pl["level"].values)
    # Perfil vertical del viento sobre un punto, a media tarde UTC: el hodógrafo detrás de la cizalladura.
    ev = cat[cat["date"].dt.date == pd.to_datetime(DAY).date()].dropna(subset=["lat", "lon"])
    lat0, lon0 = (ev.iloc[0]["lat"], ev.iloc[0]["lon"]) if not ev.empty else (-37.5, -72.0)
    t = pl["time"].values[min(21, pl.sizes["time"] - 1)]
    prof = pl.sel(time=t, latitude=lat0, longitude=lon0, method="nearest")

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].plot(prof["t"] - 273.15, prof["level"], "-o", ms=3, color="#b2182b")
    axes[0].invert_yaxis(); axes[0].set(title="Temperatura", xlabel="°C", ylabel="presión (hPa)"); axes[0].grid(alpha=0.3)
    axes[1].plot(prof["u"], prof["level"], "-o", ms=3, label="u", color="#2166ac")
    axes[1].plot(prof["v"], prof["level"], "-o", ms=3, label="v", color="#1b7837")
    axes[1].invert_yaxis(); axes[1].set(title="Viento por componente", xlabel="m/s"); axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.suptitle(f"Perfil en ({lat0:.2f}, {lon0:.2f}) · {pd.to_datetime(str(t)):%Y-%m-%d %H:%M} UTC")
    plt.tight_layout(); plt.show()

## 6. GOES multiespectral

`goes_latlon/<día>/goes_<...>.nc` — las imágenes del satélite ya reproyectadas a la misma grilla lat/lon que ERA5, con las **16 bandas** del ABI (`C01`–`C16`). Las bandas visibles e infrarrojas cercanas (`C01`–`C06`) vienen como **reflectancia** (adimensional, `1`); las infrarrojas (`C07`–`C16`) como **temperatura de brillo en K** — `C13` (10,3 µm, «ventana limpia») es la típica para ver topes de nube fríos.

> Resolución mucho más fina que ERA5 (aquí ~1301×501). Nada corrige **paralaje** todavía: a 53°S desplaza los topes ~21 km.

In [ ]:
if not HAS_GOES:
    print("No hay GOES reproyectado para este día. Corre download_goes.py / process_goes.py (ver sección 2).")
else:
    # Reunimos todas las horas del día (cada archivo trae varias) y elegimos la más tardía.
    times = []
    for r in goes_files:
        d = r.open_dataset()
        for tt in d["time"].values:
            times.append((tt, r))
    times.sort(key=lambda x: x[0])
    print(f"{len(times)} imágenes GOES el día {DAY}; de {pd.to_datetime(str(times[0][0])):%H:%M} "
          f"a {pd.to_datetime(str(times[-1][0])):%H:%M} UTC")
    t_sel, r_sel = times[-1]
    g = r_sel.open_dataset().sel(time=t_sel)
    print("bandas:", list(g.data_vars))

In [ ]:
if HAS_GOES:
    # C02 (visible, reflectancia) y C13 (IR ventana, temperatura de brillo): día y topes fríos.
    ev = cat[cat["date"].dt.date == pd.to_datetime(DAY).date()].dropna(subset=["lat", "lon"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 8))
    g["C02"].plot(ax=axes[0], cmap="gray", add_colorbar=True, cbar_kwargs={"shrink": 0.6})
    axes[0].set(title="C02 visible (reflectancia)")
    (g["C13"] - 273.15).plot(ax=axes[1], cmap="gist_ncar", add_colorbar=True, cbar_kwargs={"shrink": 0.6})
    axes[1].set(title="C13 IR ventana (°C) — topes fríos = tormenta profunda")
    for ax in axes:
        if not ev.empty:
            ax.scatter(ev["lon"], ev["lat"], c="red", marker="^", s=60, edgecolor="white", zorder=5)
    fig.suptitle(f"GOES · {pd.to_datetime(str(t_sel)):%Y-%m-%d %H:%M} UTC", y=0.98)
    plt.tight_layout(); plt.show()

## 7. Resumen y siguientes pasos

- **Catálogo**: 102 positivos, sesgados a los últimos años (mejor reporte), concentrados en el centro-sur en otoño-invierno. Sin negativos: hay que construirlos.
- **ERA5 superficie**: CAPE modesta — régimen **HSLC**, por eso importa la cizalladura, no la inestabilidad.
- **Índices**: `shear_01`, `vgp_01`, `stp_lm` (negativo) y `srh` (negativa) son los que el documento calibra para Chile. **Cuidar los signos del hemisferio sur.**
- **Niveles de presión**: perfiles verticales; enmascarar lo que queda bajo el terreno andino.
- **GOES**: 16 bandas a alta resolución; `C13` para topes fríos. Sin corrección de paralaje.

Ideas para seguir (ver `docs/mesociclones_chile.md`, sección 8, y la página *Starter kit* del visor): ordenar y limpiar el catálogo, construir negativos, animar los casos GOES conocidos, y —ya sin depender del catálogo— **buscar rotación en el yunque con flujo óptico**.